In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests, io, os
from datetime import date

plt.rcParams.update({
    'figure.facecolor': '#0C1017', 'axes.facecolor': '#111924',
    'axes.edgecolor': '#1C2A3D',   'axes.labelcolor': '#7A95B8',
    'xtick.color': '#7A95B8',      'ytick.color': '#7A95B8',
    'text.color': '#D8E4F0',       'grid.color': '#1C2A3D',
    'font.family': 'monospace',
})
BLUE, GREEN, RED, DIM = '#3B82F6', '#22C55E', '#EF4444', '#3D5470'

In [ ]:
trades = [
    {
        'id':         '001',
        'date_open':  date(2026, 10, 23),
        'date_close': None,
        'instrument': 'TTF Natural Gas (front-month)',
        'direction':  'long',
        'entry':      93.25,
        'target':     127.00,
        'stop':       80.00,
        'exit':       None,
        'notional':   100000,
        'conviction': 4,
        'status':     'open',
    },
]

df = pd.DataFrame(trades)
df[['id','instrument','direction','entry','target','stop','status']]

In [ ]:
ttf_df = pd.read_csv(os.path.expanduser('~/quantock-capital/data/ttf_prices.csv'), parse_dates=['date'])
CURRENT_TTF = float(ttf_df['price'].iloc[-1])
ttf_history = dict(zip(ttf_df['date'].dt.strftime('%Y-%m-%d'), ttf_df['price'].astype(float)))

print(f'TTF: EUR {CURRENT_TTF:.2f}/MWh  ({ttf_df["date"].iloc[-1].date()})')

In [ ]:
def fred_csv(sid):
    r = requests.get(f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}', timeout=15)
    df = pd.read_csv(io.StringIO(r.text))
    df.columns = ['date', sid]
    df['date'] = pd.to_datetime(df['date'])
    return df.set_index('date').replace('.', np.nan).dropna().astype(float)

series = {
    'DTWEXBGS':    'USD Index    ',
    'T10YIE':      'Breakeven    ',
    'DCOILBRENTEU':'Brent (EUR)  ',
    'DEXUSEU':     'EUR/USD      ',
}
macro = {}
for sid, name in series.items():
    try:
        s = fred_csv(sid)
        macro[sid] = s
        v, v2 = s[sid].iloc[-1], s[sid].iloc[-2]
        print(f'{name} {v:.2f}  {"▲" if v > v2 else "▼"} {v-v2:+.2f}  ({s.index[-1].date()})')
    except Exception as e:
        print(f'{name} failed: {e}')

In [ ]:
GIE_API_KEY = 'f12c26e577e86acabd4999c19bee2257'

if GIE_API_KEY != 'YOUR_GIE_KEY_HERE':
    r = requests.get('https://agsi.gie.eu/api?type=EU&size=10',
                     headers={'x-key': GIE_API_KEY}, timeout=10)
    data = r.json().get('data', [])
    if data:
        pct = float(data[0].get('full', 0))
        chg = pct - float(data[1].get('full', 0)) if len(data) > 1 else 0
        print(f'EU storage: {pct:.1f}%  ({chg:+.1f}pp WoW)')
else:
    MANUAL_STORAGE_PCT = 92.4
    print(f'EU storage (manual): {MANUAL_STORAGE_PCT}%')

In [ ]:
current_prices = {'001': CURRENT_TTF}

def calc(row):
    price = row['exit'] if row['status'] == 'closed' else current_prices.get(row['id'], row['entry'])
    raw = (price - row['entry']) if row['direction'] == 'long' else (row['entry'] - price)
    pct = raw / row['entry']
    return pd.Series({'price': price, 'pnl_pct': pct,
                       'pnl_eur': pct * row['notional'],
                       'r': raw / abs(row['entry'] - row['stop'])})

result = pd.concat([df[['id','instrument','direction','entry','status']], df.apply(calc, axis=1)], axis=1)

for _, r in result.iterrows():
    s = '+' if r.pnl_eur >= 0 else ''
    print(f'#{r.id}  {r.instrument}')
    print(f'     Entry: EUR {r.entry:.2f}  Current: EUR {r.price:.2f}')
    print(f'     P&L:  {s}{r.pnl_pct:.2%}  ({s}EUR {r.pnl_eur:,.0f})  |  {r.r:+.2f}R')

In [ ]:
closed = result[result['status'] == 'closed']
if len(closed) == 0:
    print('no closed trades')
else:
    rets     = closed['pnl_pct'].values
    wins     = (rets > 0).sum()
    win_rate = wins / len(rets)
    avg_w    = rets[rets > 0].mean() if wins > 0 else 0
    avg_l    = rets[rets < 0].mean() if (rets < 0).any() else 0
    sharpe   = (rets.mean() / rets.std()) * np.sqrt(52) if len(rets) > 1 else float('nan')
    cum      = np.cumprod(1 + rets)
    max_dd   = ((cum - np.maximum.accumulate(cum)) / np.maximum.accumulate(cum)).min()

    print(f'win rate:   {win_rate:.0%}  ({wins}/{len(rets)})')
    print(f'avg win:    {avg_w:+.2%}')
    print(f'avg loss:   {avg_l:+.2%}')
    print(f'expectancy: {win_rate*avg_w + (1-win_rate)*avg_l:+.2%}')
    print(f'sharpe:     {sharpe:.2f}')
    print(f'max dd:     {max_dd:.2%}')

In [ ]:
prices_sorted = sorted(ttf_history.items())
dates = pd.to_datetime([p[0] for p in prices_sorted])
vals  = np.array([p[1] for p in prices_sorted])

t   = df[df.id == '001'].iloc[0]
row = result[result.id == '001'].iloc[0]

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(dates, vals, color=BLUE, linewidth=1.8, zorder=3)
ax.scatter(dates[-1:], vals[-1:], color=BLUE, s=50, zorder=4)
ax.axhline(t.entry,  color=BLUE,  linestyle='--', lw=1, alpha=0.7, label=f'Entry  {t.entry:.2f}')
ax.axhline(t.target, color=GREEN, linestyle='--', lw=1, alpha=0.7, label=f'Target {t.target:.2f}')
ax.axhline(t.stop,   color=RED,   linestyle='--', lw=1, alpha=0.7, label=f'Stop   {t.stop:.2f}')
ax.fill_between(dates, vals, t.entry, where=vals >= t.entry, color=GREEN, alpha=0.06)
ax.fill_between(dates, vals, t.entry, where=vals < t.entry,  color=RED,   alpha=0.06)
ax.annotate(f'P&L: {row.pnl_pct:+.2%}  ({row.r:+.2f}R)',
            xy=(0.99, 0.06), xycoords='axes fraction', ha='right',
            fontsize=9, color=GREEN if row.pnl_pct >= 0 else RED)
ax.annotate('QUANTOCK CAPITAL / 001 / TTF EUR/MWh',
            xy=(0.01, 0.97), xycoords='axes fraction', ha='left', fontsize=7.5, color=DIM)
ax.set_ylabel('EUR/MWh')
ax.legend(fontsize=8, framealpha=0.15, loc='upper left', bbox_to_anchor=(0.01, 0.9))
ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))

os.makedirs(os.path.expanduser('~/quantock-capital/charts'), exist_ok=True)
out = os.path.expanduser('~/quantock-capital/charts/thesis_001.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()